# 주제 ① 사출성형 — 추가 진단 (02_diagnosis_deep)

**목적**: `01_data_quality.ipynb`에서 "추정"으로 남긴 주장을 검증하고, 다루지 않은 축(타깃 정의, 비가동 블록 정밀 정의,
시간 누수 크기, labeled↔unlabeled 관계, 자기상관, cn7·rg3 결합 가능성, 이상치 유형 규칙)을 추가로 진단한다.
심사기준 1번(데이터 이해 및 진단)에 대응하며, **모델링 경쟁이 아니라 데이터 진단**이다. 타깃 정의 비교에 쓰는 로지스틱
회귀는 성능 경쟁용이 아니라 "어떤 타깃 정의가 안정적인 학습을 가능하게 하는가"를 보기 위한 소형 실험이다.

**구성**: 계획서(`.claude/plans/2026-09-26_01_03_followup_plan.md` A-2) 번호 순.
① 쌍 구조 규명 · ② 비가동 블록 정밀 정의 · ③ 드리프트·시간 누수 · ④ labeled↔unlabeled 매칭 ·
⑤ 변수 성격 전수 점검 · ⑥ 샷 간 자기상관 · ⑦ cn7·rg3 결합 가능성 · ⑧ 이상치 유형 규칙

각 섹션은 **가설 → 실험 → 결과 → 모델 단계 반영** 순으로 서술한다.


In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import data_quality as dq
import pairs, drift, outliers
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
def save(name): plt.tight_layout(); plt.savefig(FIG / name, dpi=110); plt.close(); print("saved", name)


In [2]:
dfs = dq.load_all()
for k, d in dfs.items():
    print(f"{k:14s} shape={d.shape}")


labeled_cn7    shape=(1211, 25)
labeled_rg3    shape=(1182, 25)
unlabeled_cn7  shape=(35239, 24)
unlabeled_rg3  shape=(35941, 24)


## ① 쌍 구조 규명

**가설**: labeled의 모든 X가 정확히 2회 등장한다는 것은 01에서 확인했다. 그런데 이것이 (a) 24변수가 **완전히 동일**해서인지
아니면 (b) `row_key`가 8자리로 반올림해서 우연히 같아 보이는 것인지는 미확인이다. 또한 라벨 충돌 쌍이 특정 변수 조건에
몰려 있는지, 충돌 쌍 안에서 불량이 항상 같은 순서(첫 행 또는 둘째 행)에 오는지도 미확인이다.

**실험**: `pairs.pair_table`로 키별 그룹(쌍/예외)을 만들고 24변수 **반올림 없는 완전 일치** 여부를 직접 비교한다.
인접하지 않은 쌍의 위치를 확인한다. `pairs.mannwhitney_conflict_vs_agree`로 충돌 쌍과 일치 쌍의 변수별 분포 차이를
Mann-Whitney U로 검정한다. `pairs.order_bias_test`로 충돌 쌍 내 불량 행의 순서 편향을 이항검정한다. 마지막으로
`pairs.target_variants` + `pairs.cv_f1_by_target`으로 타깃 정의 3안(max/mean≥0.5/충돌 제외)의 양성 수와
GroupKFold(X-키) 5-fold 로지스틱(class_weight='balanced') F1 분포를 비교한다.


In [3]:
pt_cn7 = pairs.pair_table(dfs["labeled_cn7"])
pt_rg3 = pairs.pair_table(dfs["labeled_rg3"])
for name, pt in [("cn7", pt_cn7), ("rg3", pt_rg3)]:
    print(f"===== {name}")
    print("group_size:", pt["group_size"].value_counts().to_dict())
    print("pattern:", pt["pattern"].value_counts().to_dict())
    print("exact_equal(그룹크기2):", pt.loc[pt.group_size == 2, "exact_equal"].value_counts().to_dict())


===== cn7
group_size: {2: 605, 1: 1}
pattern: {'00': 591, '01': 11, '11': 3, 'single': 1}
exact_equal(그룹크기2): {True: 605}
===== rg3
group_size: {2: 591}
pattern: {'00': 566, '01': 25}
exact_equal(그룹크기2): {True: 591}


In [4]:
# 인접 여부(24변수 완전일치 쌍 기준)
for name, pt in [("cn7", pt_cn7), ("rg3", pt_rg3)]:
    p2 = pt[pt.group_size == 2]
    n_non_adj = int((p2["adjacent"] == False).sum())
    print(f"{name}: 비인접 쌍 {n_non_adj}/{len(p2)} ({n_non_adj/len(p2):.1%}), gap 분포:")
    print(p2["gap"].value_counts().sort_index().to_dict())


cn7: 비인접 쌍 35/605 (5.8%), gap 분포:
{1.0: 570, 2.0: 20, 3.0: 12, 4.0: 2, 5.0: 1}
rg3: 비인접 쌍 17/591 (2.9%), gap 분포:
{1: 574, 2: 8, 3: 9}


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, pt) in zip(axes, [("cn7", pt_cn7), ("rg3", pt_rg3)]):
    p2 = pt[pt.group_size == 2]
    ax.hist(p2["gap"], bins=np.arange(0.5, p2["gap"].max() + 1.5, 1), color="steelblue", edgecolor="white")
    ax.set_title(f"{name}: 쌍 인덱스 간격(gap) 분포"); ax.set_xlabel("gap (idx2-idx1)")
save("02_pair_gap.png")


saved

 02_pair_gap.png


In [6]:
# 비인접 쌍의 위치(파일 내 어디에 몰려 있는가)
for name, pt in [("cn7", pt_cn7), ("rg3", pt_rg3)]:
    p2 = pt[pt.group_size == 2]
    non_adj = p2[p2["adjacent"] == False].sort_values("idx1")
    if len(non_adj):
        print(f"{name}: 비인접 쌍 idx1 범위 [{non_adj['idx1'].min()}, {non_adj['idx1'].max()}] "
              f"(파일 전체 {pt['idx1'].min()}~{int(pt['idx2'].max())})")
    else:
        print(f"{name}: 비인접 쌍 없음")


cn7: 비인접 쌍 idx1 범위 [295, 1190] (파일 전체 0~1210)
rg3: 비인접 쌍 idx1 범위 [1351, 2268] (파일 전체 1211~2392)


In [7]:
print("=== 충돌 쌍 vs 일치 쌍 변수별 Mann-Whitney (cn7, p오름차순 상위 8) ===")
mw_cn7 = pairs.mannwhitney_conflict_vs_agree(dfs["labeled_cn7"], pt_cn7)
mw_cn7.head(8)


=== 충돌 쌍 vs 일치 쌍 변수별 Mann-Whitney (cn7, p오름차순 상위 8) ===


,n_conflict,n_agree,U,p,effect_r,p_bonf
var,,,,,,
Mold_Temperature_3,11,594,5851.0,0.000007,-0.790940,0.000157
Mold_Temperature_4,11,594,5822.5,0.000008,-0.782216,0.000199
Filling_Time,11,594,5668.5,0.000024,-0.735078,0.000580
Plasticizing_Position,11,594,5633.5,0.000025,-0.724365,0.000610
Injection_Time,11,594,5610.0,0.000038,-0.717172,0.000904
Max_Injection_Speed,11,594,930.0,0.000043,0.715335,0.001022
Clamp_Close_Time,11,594,5279.5,0.000056,-0.616009,0.001349
Plasticizing_Time,11,594,1141.5,0.000213,0.650597,0.005107


In [8]:
print("=== 충돌 쌍 vs 일치 쌍 변수별 Mann-Whitney (rg3, p오름차순 상위 8) ===")
mw_rg3 = pairs.mannwhitney_conflict_vs_agree(dfs["labeled_rg3"], pt_rg3)
mw_rg3.head(8)


=== 충돌 쌍 vs 일치 쌍 변수별 Mann-Whitney (rg3, p오름차순 상위 8) ===


,n_conflict,n_agree,U,p,effect_r,p_bonf
var,,,,,,
Barrel_Temperature_5,25,566,9034.0,0.018246,-0.276890,0.437907
Plasticizing_Time,25,566,7956.0,0.290612,-0.124523,1.000000
Cycle_Time,25,566,7747.0,0.351558,-0.094982,1.000000
Clamp_Close_Time,25,566,7738.5,0.387304,-0.093781,1.000000
Barrel_Temperature_3,25,566,6375.0,0.397362,0.098940,1.000000
Hopper_Temperature,25,566,6429.0,0.439325,0.091307,1.000000
Max_Screw_RPM,25,566,7626.0,0.473980,-0.077880,1.000000
Average_Screw_RPM,25,566,7513.0,0.551249,-0.061908,1.000000


In [9]:
print("cn7 Bonferroni 유의(p_bonf<0.05) 변수 수:", int((mw_cn7["p_bonf"] < 0.05).sum()), "/", len(mw_cn7))
print("rg3 Bonferroni 유의(p_bonf<0.05) 변수 수:", int((mw_rg3["p_bonf"] < 0.05).sum()), "/", len(mw_rg3))


cn7 Bonferroni 유의(p_bonf<0.05) 변수 수: 10 / 24
rg3 Bonferroni 유의(p_bonf<0.05) 변수 수: 0 / 24


In [10]:
print("=== 충돌 쌍 내 불량 행 순서 편향 ===")
ob_cn7 = pairs.order_bias_test(pt_cn7)
ob_rg3 = pairs.order_bias_test(pt_rg3)
print("cn7:", ob_cn7)
print("rg3:", ob_rg3)


=== 충돌 쌍 내 불량 행 순서 편향 ===
cn7: {'n_conflict': 11, 'fail_first': 10, 'fail_second': 1, 'binom_p': np.float64(0.01171875)}
rg3: {'n_conflict': 25, 'fail_first': 23, 'fail_second': 2, 'binom_p': np.float64(1.9431114196777344e-05)}


**결과**
- 24변수를 반올림 없이 비교해도 모든 쌍이 **완전 일치**한다(cn7 605쌍, rg3 591쌍 전부 `exact_equal=True`). 8자리 반올림
  키가 우연히 같아 보이는 경우는 없다 → 진짜 "같은 입력이 2행으로 기록"된 구조다.
- 비인접 쌍 비율은 cn7 5.8%(35/605, gap 최대 5), rg3는 전부 인접(0%)이다. cn7 비인접 쌍은 특정 구간(행
  약 295~1192)에 몰려 있고, 서로 다른 두 쌍의 인덱스가 교차(예: 401·403 쌍과 402·404 쌍)하는 형태 → 그 구간에서
  기록 순서가 인터리브된 것으로 추정.
- 충돌 쌍 vs 일치 쌍 Mann-Whitney: **cn7은 Bonferroni 보정 후에도 24개 중 10개 변수**(주로 `Mold_Temperature_3/4`,
  `Filling_Time`, `Plasticizing_Position`, `Injection_Time`)**가 유의(p_bonf<0.05)**하다 → cn7 충돌 쌍은 무작위로
  흩어진 게 아니라 드리프트 구간(③ 참고)에 몰려 있다는 뜻. **rg3는 24개 중 유의한 변수가 0개**(최소 p_bonf=0.44) →
  rg3 충돌은 특정 조건과 무관하게 전 구간에 고르게 나타나는 순수한 라벨 노이즈에 가깝다.
- 순서 편향: 충돌 쌍에서 불량이 **첫 번째 행에 오는 경우가 압도적으로 많다**(cn7 10/11, 이항검정 p≈0.012; rg3 23/25,
  p≈2e-05). 우연이라면 50%에 가까워야 하는데 크게 벗어난다 → "먼저 기록된 샷이 불합격, 재검사/재작업 후 두 번째
  기록이 합격"인 수집 관행일 가능성이 높다(추정). 이는 3안(충돌 제외) 외에 **"첫 행 라벨을 신뢰"하는 4번째 타깃 정의
  후보**를 시사한다.

**모델 단계 반영**: cn7은 GroupKFold와 별개로 **시간 블록 split을 반드시 병행**해야 한다(충돌이 드리프트 구간에
몰려 있어 랜덤 split은 이 구간을 학습에도 노출시켜 과대평가를 유발). rg3는 조건-무관 라벨 노이즈이므로 피처 튜닝보다
**라벨 신뢰도 자체를 모델 성능 상한으로 보고**해야 한다. 순서 편향은 참고용 가설로 리포트에만 남기고(표본 11·25건으로
추가 검증 없이 규칙화하기엔 근거가 약함), 실제 타깃 정의는 아래 F1 비교로 결정한다.


In [11]:
variants_cn7, meta_cn7 = pairs.target_variants(dfs["labeled_cn7"])
variants_rg3, meta_rg3 = pairs.target_variants(dfs["labeled_rg3"])
print("cn7 meta:", meta_cn7)
print("rg3 meta:", meta_rg3)
summary = []
for name, files in [("cn7", variants_cn7), ("rg3", variants_rg3)]:
    for tname, (X, y, g) in files.items():
        summary.append({"file": name, "target": tname, "n_rows": len(y), "n_pos": int(y.sum())})
pd.DataFrame(summary)


cn7 meta: {'group_pos': 14, 'group_soft_pos': 8.5, 'group_pure_pos': 3, 'n_groups': 606, 'n_conflict_groups': 11}
rg3 meta: {'group_pos': 25, 'group_soft_pos': 12.5, 'group_pure_pos': 0, 'n_groups': 591, 'n_conflict_groups': 25}


,file,target,n_rows,n_pos
0,cn7,max,1211,28
1,cn7,mean_ge_0.5,1211,28
2,cn7,exclude_conflict,1189,6
3,rg3,max,1182,50
4,rg3,mean_ge_0.5,1182,50
5,rg3,exclude_conflict,1132,0


In [12]:
f1_cn7 = pairs.cv_f1_by_target(dfs["labeled_cn7"])
f1_rg3 = pairs.cv_f1_by_target(dfs["labeled_rg3"])
print("=== cn7 F1 (target별 median/mean/std/count) ===")
display(f1_cn7.groupby("target")["f1"].agg(["median", "mean", "std", "count"]))
print("=== rg3 F1 ===")
display(f1_rg3.groupby("target")["f1"].agg(["median", "mean", "std", "count"]))
print(f1_rg3.loc[f1_rg3["note"] != "", ["target", "note"]].drop_duplicates())


=== cn7 F1 (target별 median/mean/std/count) ===


,median,mean,std,count
target,,,,
exclude_conflict,1.000000,0.600000,0.547723,5
max,0.153846,0.184615,0.200591,5
mean_ge_0.5,0.153846,0.184615,0.200591,5


=== rg3 F1 ===


,median,mean,std,count
target,,,,
exclude_conflict,NaN,NaN,NaN,0
max,0.078431,0.060958,0.059168,5
mean_ge_0.5,0.078431,0.060958,0.059168,5


              target                    note
10  exclude_conflict  양성 또는 음성 표본 0개 - 실행 불가


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, f1df) in zip(axes, [("cn7", f1_cn7), ("rg3", f1_rg3)]):
    d = f1df.dropna(subset=["f1"])
    if len(d):
        sns.stripplot(data=d, x="target", y="f1", ax=ax, size=8, color="steelblue")
        sns.boxplot(data=d, x="target", y="f1", ax=ax, showcaps=True, boxprops={"facecolor": "none"}, showfliers=False)
    ax.set_title(f"{name}: 타깃 정의별 GroupKFold F1"); ax.set_ylim(-0.05, 1.05)
    ax.tick_params(axis="x", rotation=15)
save("02_target_f1_dist.png")


saved 02_target_f1_dist.png


**결과**
- 그룹 크기가 항상 1~2이므로 `max`와 `mean≥0.5`는 **이진 결정이 수학적으로 동일**하다(둘 다 충돌 쌍을 양성으로 만든다).
  차이는 "기대 양성 수"에서만 드러난다: cn7 그룹 단위 고유 불량 14개(=max 기준) vs 8.5개(=mean 기준, 충돌을 0.5로
  카운트), rg3는 25개 vs 12.5개.
- `충돌 제외`는 rg3에서 **양성이 0개가 되어 분류 자체가 불가능**하다(rg3 불량 25개 전부가 충돌 쌍이므로). cn7은
  양성 6행(순수 불량 그룹 3개×2행)만 남아 5-fold 중 다수 fold에 양성이 1개뿐이라 F1이 fold마다 0 또는 1로 극단적으로
  흔든다(median 1.0, mean 0.60, std 0.55, n=5).
- `max`(=`mean≥0.5`) 기준 F1은 cn7 median 0.154(mean 0.185, std 0.20), rg3 median 0.078(mean 0.061, std 0.059) —
  두 정의 모두 절대 수준이 낮고 fold 간 편차가 크다.

**모델 단계 반영**: (1) rg3는 `충돌 제외`를 쓸 수 없다(양성 0) → 반드시 `max`(불량 의심 포함) 정의를 쓰거나, 충돌
쌍을 "판정 불가(모델 평가 제외, 학습에는 약한 양성으로만 사용)"로 별도 처리해야 한다. (2) cn7의 `충돌 제외` F1이
높아 보이는 것은 **표본이 너무 적어(양성 6개) 나온 착시**이므로 "충돌 제외가 더 좋은 정의"라고 해석하면 안 된다 —
표에 그대로 신뢰구간 없이 보고하지 않는다. (3) 모델 단계 기본값은 **`max` 정의 + GroupKFold(X-키) + 시간 블록 split
병행**으로 확정하고, `충돌 제외`는 "라벨이 확실한 샷만의 상한선 참고용"으로만 부기한다.


## ④ labeled <-> unlabeled 근사 매칭

**가설**: labeled와 unlabeled는 스케일이 달라(파일별 z-score) 값으로 직접 비교할 수 없다. 원본 인덱스 범위도 겹치지
않는다(01에서 확인). 그러나 **같은 샷이 두 파일에 모두 기록**됐다면(예: labeled가 unlabeled의 부분집합을 다시
표준화한 것이라면) unlabeled를 사전학습·이상탐지에 쓸 때 test 행이 섞이는 누수가 생긴다.

**실험**: 파일 내부 순위(percentile rank)로 정규화한 24차원 벡터로 labeled 각 행의 unlabeled 최근접 이웃 거리를
구하고, 완전 무작위 벡터의 최근접 거리(기준선)와 비교한다(`pairs.rank_match_to_reference`).


In [14]:
match_cn7, rand_cn7 = pairs.rank_match_to_reference(dfs["labeled_cn7"], dfs["unlabeled_cn7"])
match_rg3, rand_rg3 = pairs.rank_match_to_reference(dfs["labeled_rg3"], dfs["unlabeled_rg3"])
from scipy.stats import ks_2samp
for name, m, r in [("cn7", match_cn7, rand_cn7), ("rg3", match_rg3, rand_rg3)]:
    stat, p = ks_2samp(m["dist"], r)
    print(f"===== {name}")
    print("labeled->unlabeled 최근접거리:", m["dist"].describe().round(3).to_dict())
    print("무작위 벡터 최근접거리(기준선):", pd.Series(r).describe().round(3).to_dict())
    print(f"KS(매칭거리 vs 무작위거리): stat={stat:.4f} p={p:.3g}")


===== cn7
labeled->unlabeled 최근접거리: {'count': 1211.0, 'mean': 1.309, 'std': 0.137, 'min': 0.841, '25%': 1.224, '50%': 1.307, '75%': 1.401, 'max': 1.897}
무작위 벡터 최근접거리(기준선): {'count': 1211.0, 'mean': 1.404, 'std': 0.154, 'min': 0.777, '25%': 1.313, '50%': 1.415, '75%': 1.51, 'max': 1.834}
KS(매칭거리 vs 무작위거리): stat=0.3039 p=9.53e-50
===== rg3
labeled->unlabeled 최근접거리: {'count': 1182.0, 'mean': 1.281, 'std': 0.164, 'min': 0.825, '25%': 1.166, '50%': 1.29, '75%': 1.403, 'max': 1.704}
무작위 벡터 최근접거리(기준선): {'count': 1182.0, 'mean': 1.387, 'std': 0.161, 'min': 0.825, '25%': 1.279, '50%': 1.398, '75%': 1.502, 'max': 1.854}
KS(매칭거리 vs 무작위거리): stat=0.2623 p=3.9e-36


In [15]:
eps = 0.3
for name, m in [("cn7", match_cn7), ("rg3", match_rg3)]:
    share = float((m["dist"] < eps).mean())
    print(f"{name}: dist<{eps} 인 labeled 행 비율 = {share:.4f} ({int((m['dist']<eps).sum())}/{len(m)})")


cn7: dist<0.3 인 labeled 행 비율 = 0.0000 (0/1211)
rg3: dist<0.3 인 labeled 행 비율 = 0.0000 (0/1182)


**결과**
- labeled→unlabeled 최근접거리 분포(cn7 mean 1.31·min 0.84, rg3 mean 1.28·min 0.83)는 **무작위 벡터의 최근접거리
  분포(cn7 mean 1.40·min 0.78, rg3 mean 1.39·min 0.82)와 거의 같은 범위**에 있고, 오히려 labeled의 최솟값이 무작위
  기준선보다 크거나 비슷하다. KS 검정은 두 분포가 통계적으로 다르다고 나오지만(cn7 stat=0.304, rg3 stat=0.262,
  p≪0.001 — 표본이 커서 작은 차이도 유의하게 나옴) **"거의 완전히 같은 샷"에 해당하는 근접 매칭(ε=0.3)은 cn7·rg3
  모두 0건**이다.
- 즉 **값 기준으로 labeled 행이 unlabeled에 재등장하지 않는다**는 01의 "원본 인덱스가 겹치지 않는다" 관찰이
  근사 매칭으로도 재확인됐다. unlabeled를 사전학습·이상탐지에 쓰더라도 labeled test 행이 값으로 섞여 들어가는
  누수는 (이 근사 기준 안에서는) 없다.

**모델 단계 반영**: unlabeled를 사전학습(오토인코더 등)·이상탐지·드리프트 참고용으로 자유롭게 사용해도 **labeled
평가셋 누수 위험은 낮다**고 판단한다. 다만 이 결론은 "순위 벡터 유클리드 거리 ε=0.3" 기준에서만 유효한 (추정)이므로,
모델 단계에서 실제로 unlabeled 사전학습을 쓴다면 사전학습 전후 labeled CV 성능 차이를 별도로 재확인하는 것을
권장한다.


## ⑤ 변수 성격 전수 점검

**가설**: 01은 `Clamp_Open_Position`이 상수, rg3의 `Injection_Time`(3값)·`Filling_Time`(2값)이 사실상 이산이라고
지적했지만 24변수 × 4파일 전수 점검(고유값 수, 이산/연속, 스파이크 값)은 하지 않았다. rg3 두 변수의 "전환 시점"도
확인이 필요하다.

**실험**: `pairs.var_profile`로 4파일 전체를 점검하고, `pairs.value_segments`로 rg3 `Injection_Time`·`Filling_Time`의
행 순서상 값 전환 구간을 찾아 구간별 불량률을 비교한다.


In [16]:
profiles = {k: pairs.var_profile(dq.split_xy(d)[0]) for k, d in dfs.items()}
n_spike = pd.DataFrame({k: p["is_spike"] for k, p in profiles.items()})
n_discrete = pd.DataFrame({k: p["is_discrete"] for k, p in profiles.items()})
print("=== 파일별 스파이크(top_share>=5%) 변수 개수 ===")
print(n_spike.sum().to_dict())
print("=== 파일별 이산(nunique<=20) 변수 개수 ===")
print(n_discrete.sum().to_dict())
print("=== 상수 변수(nunique==1) ===")
for k, p in profiles.items():
    const_vars = p.index[p["is_constant"]].tolist()
    print(k, const_vars)


=== 파일별 스파이크(top_share>=5%) 변수 개수 ===
{'labeled_cn7': 23, 'labeled_rg3': 23, 'unlabeled_cn7': 22, 'unlabeled_rg3': 23}
=== 파일별 이산(nunique<=20) 변수 개수 ===
{'labeled_cn7': 15, 'labeled_rg3': 15, 'unlabeled_cn7': 0, 'unlabeled_rg3': 0}
=== 상수 변수(nunique==1) ===
labeled_cn7 ['Clamp_Open_Position']
labeled_rg3 ['Clamp_Open_Position']
unlabeled_cn7 []
unlabeled_rg3 []


In [17]:
print("=== rg3 labeled: 이산(고유값<=10) 변수 프로필 ===")
display(profiles["labeled_rg3"][profiles["labeled_rg3"]["nunique"] <= 10].sort_values("nunique"))


=== rg3 labeled: 이산(고유값<=10) 변수 프로필 ===


,nunique,is_constant,is_discrete,top_value,top_share,is_spike
var,,,,,,
Clamp_Open_Position,1,True,True,0.000000,1.000000,False
Filling_Time,2,False,True,0.838552,0.587140,True
Clamp_Close_Time,3,False,True,-0.386790,0.429780,True
Injection_Time,3,False,True,-0.029099,0.986464,True
Average_Screw_RPM,4,False,True,-0.421798,0.583756,True
Max_Screw_RPM,4,False,True,-0.665752,0.428088,True
Max_Injection_Speed,7,False,True,0.617645,0.270728,True
Cushion_Position,7,False,True,0.480326,0.282572,True
Plasticizing_Position,8,False,True,0.588367,0.302876,True


In [18]:
X_rg3_, y_rg3_seg = dq.split_xy(dfs["labeled_rg3"])
seg_it = pairs.value_segments(X_rg3_["Injection_Time"])
seg_ft = pairs.value_segments(X_rg3_["Filling_Time"])
print(f"Injection_Time: 세그먼트 {len(seg_it)}개, 값 {seg_it['value'].unique()}")
print(f"Filling_Time: 세그먼트 {len(seg_ft)}개, 값 {seg_ft['value'].unique()}")
print(seg_it.to_string())


Injection_Time: 세그먼트 15개, 값 [-0.02909945 -8.62773039  8.56963399]
Filling_Time: 세그먼트 108개, 값 [-1.1925314   0.83855234]
    start_pos  end_pos     value  length
0           0       67 -0.029099      68
1          68       71 -8.627730       4
2          72      399 -0.029099     328
3         400      401 -8.627730       2
4         402      953 -0.029099     552
5         954      955  8.569634       2
6         956      967 -0.029099      12
7         968      969  8.569634       2
8         970     1021 -0.029099      52
9        1022     1023  8.569634       2
10       1024     1039 -0.029099      16
11       1040     1041  8.569634       2
12       1042     1115 -0.029099      74
13       1116     1117  8.569634       2
14       1118     1181 -0.029099      64


In [19]:
# Injection_Time: 주값(최다 세그먼트) vs 희귀값 세그먼트의 불량률
main_val = seg_it["value"].mode().iloc[0]
rare = seg_it[seg_it["value"] != main_val]
rows = []
for _, r in seg_it.iterrows():
    sl = y_rg3_seg.iloc[int(r.start_pos):int(r.end_pos) + 1]
    rows.append({"value": r.value, "length": r.length, "n_fail": int(sl.sum()), "fail_rate": float(sl.mean())})
seg_fail = pd.DataFrame(rows)
print("=== Injection_Time 값별(세그먼트 단위) 불량률 ===")
display(seg_fail.groupby("value").agg(n_segments=("length", "size"), total_rows=("length", "sum"),
                                       total_fail=("n_fail", "sum")).assign(
    fail_rate=lambda d: d["total_fail"] / d["total_rows"]))


=== Injection_Time 값별(세그먼트 단위) 불량률 ===


,n_segments,total_rows,total_fail,fail_rate
value,,,,
-8.627730,2,6.0,0,0.000000
-0.029099,8,1166.0,25,0.021441
8.569634,5,10.0,0,0.000000


**결과**
- 파일별 스파이크(한 값이 5% 이상 차지) 변수 수: labeled cn7 23개, labeled rg3 23개, unlabeled cn7 22개, unlabeled
  rg3 23개(24개 중 대부분이 스파이크 후보). z-score 표준화 데이터에서 스파이크가 흔한 것은 **셋포인트 제어(사출
  공정은 기본적으로 설정값을 반복 재현하려 하므로 같은 값이 자주 나옴)**로 설명 가능(추정) — 반드시 숨은 결측을
  의미하지는 않는다.
- 이산 변수(고유값 <=20)로 분류되는 변수는 **labeled cn7·rg3 둘 다 24개 중 15개**(`Injection_Time`, `Filling_Time`,
  `Cycle_Time`, `Clamp_Close_Time`, `Cushion_Position`, `Plasticizing_Position`, `Clamp_Open_Position`,
  `Max_Injection_Speed`, `Max_Screw_RPM`, `Average_Screw_RPM`, `Max_Injection_Pressure`, `Max_Switch_Over_Pressure`,
  `Barrel_Temperature_3`, `Barrel_Temperature_6` + cn7는 `Average_Back_Pressure`/rg3는 `Barrel_Temperature_1`)로
  **동일**하다. 반면 **unlabeled는 두 파일 다 이산 변수가 0개**(모든 변수 고유값 >20) — labeled의 "이산성"이
  unlabeled에서는 재현되지 않는다. 행 수 차이(1,200 대 35,000)만으로 설명하기엔 차이가 너무 크므로, labeled가
  unlabeled보다 **좁은 설정값 범위(또는 다른 수집 조건)**에서 모인 것이라는 기존 가설(②의 idle 블록 부재와 같은
  맥락)을 추가로 뒷받침한다(추정).
- `Clamp_Open_Position`은 4개 파일 전부에서 상수(nunique=1) — 완전한 상수이므로 모델링에서 제거 대상 1순위로 확정.
- rg3 `Injection_Time`은 "단일 전환 시점"이 아니라 **주값(-0.029, top_share 98.6%)과 희귀값(-8.63/+8.57)이 총
  15개 세그먼트(주값 8개 + 희귀값 7개: -8.63 구간 2개·+8.57 구간 5개)로 자주 오가는 반복적 단주기 이벤트**(희귀값
  세그먼트는 대부분 길이 2~4행)다. 당초 가정("전환 전후 불량률 비교")은 이 데이터 구조와 맞지 않아 **세그먼트
  단위 불량률**로 대체 확인했다: 주값(-0.029) 구간 불량률 2.14%(25건/1,166행), 희귀값 구간은 두 값 모두
  **불량 0건**(-8.63: 0/6행, +8.57: 0/10행, 합쳐서 0/16행). `Filling_Time`도 유사하게 108개의 짧은 세그먼트로
  자주 전환된다(단일 전환점 없음).

**모델 단계 반영**: (1) `Clamp_Open_Position`은 4파일 공통 제거. (2) rg3의 `Injection_Time`/`Filling_Time` 희귀값은
불량과 무관(오히려 불량 0건)하므로 "이상치"로 취급해 제거하지 말고, **희귀값 자체를 이진 플래그 변수**(예:
"저해상도 셋포인트 전환 샷 여부")로 남겨 모델이 활용할지 자체적으로 판단하게 한다. (3) 스파이크 변수 다수는
트리 기반 모델(구간 분할에 강함)에는 문제되지 않지만 로지스틱 등 선형 모델에서는 이산화(원-핫 또는 구간화)를
검토한다.
